In [0]:
from pyspark.sql import functions as F

# 1. Caminhos de leitura (Silver) e escrita (Gold)
path_silver = "dbfs:/Volumes/workspace/default/data/delta/silver/"
path_gold = "dbfs:/Volumes/workspace/default/data/delta/gold/"

# 2. Carregar as tabelas necessárias da Silver
# Precisamos da consolidada para valores e da reviews para a nota média
orders_consolidated = spark.read.format("delta").load(f"{path_silver}orders_consolidated")
reviews = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data/delta/bronze/reviews") # Reviews costuma vir da Bronze se não houver tratamento complexo

# 3. Processamento dos KPIs por Cliente
# Conforme imagem c3a57b.png: total_revenue = price + freight_value
gold_customer_summary = orders_consolidated.groupBy("customer_id", "customer_city", "customer_state").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.sum(F.col("price") + F.col("freight_value")).alias("total_revenue"),
    F.avg(F.col("price") + F.col("freight_value")).alias("avg_order_value")
)

# 4. Join com Reviews para pegar a nota média (avg_review_score)
# Primeiro agrupamos as reviews por order_id para evitar duplicar linhas no join
reviews_avg = reviews.groupBy("order_id").agg(F.avg("review_score").alias("avg_review_score"))

# Join final para adicionar a nota média ao sumário do cliente
gold_final = gold_customer_summary.join(
    orders_consolidated.select("customer_id", "order_id").distinct(), 
    "customer_id", 
    "inner"
).join(
    reviews_avg, 
    "order_id", 
    "left"
).groupBy("customer_id", "customer_city", "customer_state", "total_orders", "total_revenue", "avg_order_value") \
 .agg(F.avg("avg_review_score").alias("avg_review_score"))

# 5. Salvamento na Camada Gold
gold_final.write.format("delta").mode("overwrite").save(f"{path_gold}customer_summary")

print("✨ Camada Gold: Tabela 'customer_summary' gerada com sucesso!")

✅ Tabela 'orders_consolidated' criada com sucesso com todas as colunas de auditoria renomeadas!
